In [1]:
import pandas as pd
import numpy as np
import torch
from  torch.optim import AdamW, Adam, SGD, RMSprop
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments, get_linear_schedule_with_warmup
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import gc
from math import ceil
from transformers import EarlyStoppingCallback
from utils.config import *

In [2]:
old_dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'
new_dataset_path= 'Model_dataset/real_world_data.csv'

# q type classifer,
q_type_model_path= 'model/fine_tuned_question_classifier_model_lite-SGD'
q_type_model_result= '.temp/model_results/q_types_model_lite_results'
q_type_model= '.temp/model/fine_tuned_question_classifier_model_lite'



# Question type classsifier

### Preprocessing

In [3]:
data_ratio= {
    "old":30,
    "new":70
    }

new_data_df= pd.read_csv(new_dataset_path)
new_data_df= new_data_df[["question", "question_type"]]
new_data_df.info()

old_data_df= pd.read_csv(old_dataset_path)
old_data_df= old_data_df[["question", "question_type"]]
old_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1330 entries, 0 to 1329
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1330 non-null   object
 1   question_type  1330 non-null   object
dtypes: object(2)
memory usage: 20.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1644 entries, 0 to 1643
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1644 non-null   object
 1   question_type  1644 non-null   object
dtypes: object(2)
memory usage: 25.8+ KB


In [4]:
columns_in_new_df= new_data_df["question_type"].unique()
print(f"columns_in_new_df :{columns_in_new_df}")

columns_in_old_df= old_data_df["question_type"].unique()
print(f"columns_in_old_df :{columns_in_old_df}")

columns_in_new_df :['personal_information' 'skills' 'availability' 'current_ctc'
 'expected_ctc' 'working_experience' 'others' 'education']
columns_in_old_df :['current_ctc' 'expected_ctc' 'personal_information' 'education'
 'working_experience' 'skills' 'availability' 'others']


In [5]:
new_data_df.drop_duplicates(inplace= True)
new_data_df.info()

old_data_df.drop_duplicates(inplace= True)
old_data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1314 entries, 0 to 1329
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1314 non-null   object
 1   question_type  1314 non-null   object
dtypes: object(2)
memory usage: 30.8+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 1634 entries, 0 to 1643
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1634 non-null   object
 1   question_type  1634 non-null   object
dtypes: object(2)
memory usage: 38.3+ KB


#### combaing new and old data

In [6]:
new_data_len= len(new_data_df)
total_len= (new_data_len/(data_ratio["new"]/100))
old_data_len= ceil(total_len- new_data_len)
print(total_len)
old_data_len

1877.1428571428573


564

In [7]:
temp_df= pd.DataFrame()
while True:
    temp_df= old_data_df.sample(old_data_len)
    columns_in_old_df= temp_df["question_type"].unique()

    if set(columns_in_old_df)== set(columns_in_new_df):
        break
temp_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 564 entries, 1529 to 926
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       564 non-null    object
 1   question_type  564 non-null    object
dtypes: object(2)
memory usage: 13.2+ KB


In [8]:
df= pd.concat([new_data_df, temp_df], ignore_index=True)
df.drop_duplicates(inplace= True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1876 entries, 0 to 1877
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1876 non-null   object
 1   question_type  1876 non-null   object
dtypes: object(2)
memory usage: 44.0+ KB


#### adding labels

In [9]:
# adding labels
label_mapping = {key: index for index, key in enumerate(QUESTION_TYPES)}
df['label'] = df['question_type'].map(label_mapping)

# droping unused column
df.drop('question_type', axis=1,  inplace= True)
df.head()

,question,label
0,Email address,2
1,Phone country code,2
2,Mobile phone number,2
3,How many years of total exp you have ?,5
4,What is your notice period in days ?,6


In [10]:
df= df.sample(frac=1).reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1876 entries, 0 to 1875
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  1876 non-null   object
 1   label     1876 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 29.4+ KB


In [11]:
df.head(5)

,question,label
0,"We are looking for Immediate joiners , Can you...",6
1,How many years of work experience do you have ?,5
2,Would you commit to delivering key milestones ...,6
3,Would you prioritize this offer over others du...,6
4,Please mention your current total compensation...,0


### Retraing Preparations:

In [12]:
#convert to hugging face dataset
dataset= Dataset.from_pandas(df)

#Split the data into train and test sets (80-20 split)
dataset_split = dataset.train_test_split(test_size=0.2)

# Access train and test splits
train_dataset = dataset_split['train']
test_dataset = dataset_split['test']

In [13]:
tokenizer = DistilBertTokenizerFast.from_pretrained(q_type_model_path)

In [14]:
def tokenize_function(examples):
    return tokenizer(examples['question'], padding= "max_length", truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set the format to PyTorch tensors
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/376 [00:00<?, ? examples/s]

In [15]:
# Mapping lebel and id
id2label = {v: k for k, v in label_mapping.items()}  # Map IDs to label names
label2id = {k: v for k, v in label_mapping.items()}  # Map label names to IDs

In [16]:
total_training_steps = (len(train_dataset) // (16 * 2)) * 4  # Example calculation: adjust as needed
warmup_steps = int(0.1 * total_training_steps)

# Retraning

# using default optimizer

In [17]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

8


In [18]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-default",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [19]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.339500,0.315598,0.901596
2,0.206900,0.342912,0.898936
3,0.185800,0.321912,0.906915


TrainOutput(global_step=141, training_loss=0.27198073621971386, metrics={'train_runtime': 4426.4833, 'train_samples_per_second': 16.943, 'train_steps_per_second': 0.531, 'total_flos': 596167077888000.0, 'train_loss': 0.27198073621971386, 'epoch': 3.0})

- seems like epoch 3 will be best for prediction

### Model evaluation and Saving

In [20]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.3155979514122009,
 'eval_accuracy': 0.901595744680851,
 'eval_runtime': 138.5087,
 'eval_samples_per_second': 2.715,
 'eval_steps_per_second': 0.173,
 'epoch': 3.0}

In [21]:
trainer.save_model(q_type_model+"-default")
tokenizer.save_pretrained(q_type_model+"-default")

('.temp/model/fine_tuned_question_classifier_model_lite-default/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-default/tokenizer.json')

In [22]:
del trainer, model

# Using AdamW optimiser with linear scheduler with warmup

In [23]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

8


In [24]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-AdamW",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [25]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = AdamW(model.parameters(), lr=5e-5, eps=1e-8)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [26]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.375200,0.350783,0.896277
2,0.322200,0.348440,0.896277
3,0.245900,0.284641,0.909574
4,0.201300,0.309015,0.912234
5,0.144700,0.324302,0.888298


TrainOutput(global_step=235, training_loss=0.29150152967331255, metrics={'train_runtime': 7484.7136, 'train_samples_per_second': 10.02, 'train_steps_per_second': 0.314, 'total_flos': 993611796480000.0, 'train_loss': 0.29150152967331255, 'epoch': 5.0})

- seems like epoch 4 will be best for prediction

In [27]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.28464144468307495,
 'eval_accuracy': 0.9095744680851063,
 'eval_runtime': 131.6462,
 'eval_samples_per_second': 2.856,
 'eval_steps_per_second': 0.182,
 'epoch': 5.0}

In [28]:
trainer.save_model(q_type_model+"-AdamW")
tokenizer.save_pretrained(q_type_model+"-AdamW")

('.temp/model/fine_tuned_question_classifier_model_lite-AdamW/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-AdamW/tokenizer.json')

In [29]:
del trainer, model


# Using Adam

In [30]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

8


In [31]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-Adam",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [32]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = Adam(model.parameters(), lr=3e-5)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [33]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.412700,0.371574,0.890957
2,0.353500,0.367845,0.890957
3,0.275200,0.298037,0.906915
4,0.233500,0.295816,0.904255
5,0.180300,0.294386,0.904255
6,0.127700,0.299964,0.914894
7,0.144200,0.327166,0.904255


TrainOutput(global_step=329, training_loss=0.2681355659360219, metrics={'train_runtime': 9995.5868, 'train_samples_per_second': 7.503, 'train_steps_per_second': 0.235, 'total_flos': 1391056515072000.0, 'train_loss': 0.2681355659360219, 'epoch': 7.0})

- seems like epoch 6 will be best for prediction

In [34]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.29438626766204834,
 'eval_accuracy': 0.9042553191489362,
 'eval_runtime': 128.503,
 'eval_samples_per_second': 2.926,
 'eval_steps_per_second': 0.187,
 'epoch': 7.0}

In [35]:
trainer.save_model(q_type_model+"-Adam")
tokenizer.save_pretrained(q_type_model+"-Adam")

('.temp/model/fine_tuned_question_classifier_model_lite-Adam/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-Adam/tokenizer.json')

In [36]:
del trainer, model


# Using SGD

In [37]:
# Loading the pre trained model
model = DistilBertForSequenceClassification.from_pretrained(
        q_type_model_path,
        num_labels= len(columns_in_old_df),
        ignore_mismatched_sizes=True,  # Allows resizing of classification head
        id2label=id2label,
        label2id=label2id,
    )
print(model.config.num_labels)  # Should print 7

8


In [38]:
training_args = TrainingArguments(
    output_dir=q_type_model_result + "-SGD",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    num_train_epochs= 50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    load_best_model_at_end=True,
)

In [39]:
# AdamW optimizer is automatically used by Hugging Face, but you can explicitly define it
optimizer = SGD(model.parameters(), lr=0.01, momentum=0.9)

# Define the linear scheduler with warmup
total_steps = len(train_dataset) * training_args.num_train_epochs // training_args.per_device_train_batch_size
warmup_steps = int(total_steps * 0.1)  # 10% of total steps for warmup
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

In [40]:
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # The training dataset
    eval_dataset=test_dataset,           # The test dataset
    optimizers=(optimizer, lr_scheduler),   # Pass optimizer and scheduler as a tuple
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    compute_metrics=lambda p: {'accuracy': accuracy_score(p.predictions.argmax(axis=-1), p.label_ids)}  # Compute accuracy during eval
)


# Clearing memory before start traning
torch.cuda.empty_cache()
gc.collect()

# Start training
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.398600,0.358277,0.893617
2,0.365300,0.388839,0.888298
3,0.293600,0.296182,0.904255
4,0.254600,0.298032,0.904255
5,0.204000,0.296170,0.901596
6,0.174000,0.285469,0.901596
7,0.200600,0.336795,0.885638
8,0.153400,0.322979,0.890957


TrainOutput(global_step=376, training_loss=0.28112272315836967, metrics={'train_runtime': 11259.3514, 'train_samples_per_second': 6.661, 'train_steps_per_second': 0.209, 'total_flos': 1589778874368000.0, 'train_loss': 0.28112272315836967, 'epoch': 8.0})

- seems like epoch 6 will be best for prediction

In [41]:
evaluation_results = trainer.evaluate()
evaluation_results

{'eval_loss': 0.28546929359436035,
 'eval_accuracy': 0.901595744680851,
 'eval_runtime': 125.5341,
 'eval_samples_per_second': 2.995,
 'eval_steps_per_second': 0.191,
 'epoch': 8.0}

In [42]:
trainer.save_model(q_type_model+"-SGD")
tokenizer.save_pretrained(q_type_model+"-SGD")

('.temp/model/fine_tuned_question_classifier_model_lite-SGD/tokenizer_config.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/special_tokens_map.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/vocab.txt',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/added_tokens.json',
 '.temp/model/fine_tuned_question_classifier_model_lite-SGD/tokenizer.json')

In [43]:
del trainer, model
